# LangGraph `wrap_tool_call`：拦截、改写与控制工具执行

本教程直接使用 LangGraph 的 `StateGraph` 与 `ToolNode`，不使用 `langchain.agents.create_agent`。主示例由真实 **DeepSeek Flash** 生成工具调用；天气工具本身是本地确定性函数，因此可以把注意力集中在 `wrap_tool_call` 的执行语义上。

学习目标：

- 理解 `ToolNode(..., wrap_tool_call=wrapper)` 在工具执行链中的位置；
- 观察 `ToolCallRequest` 的 `tool_call`、`tool`、`state` 与 `runtime`；
- 掌握 `request.override(...)`、短路、重试和结果后处理；
- 明确多次调用 `execute()` 的副作用与 `ToolMessage.tool_call_id` 协议。


## 总体流程

```mermaid
flowchart LR
    S([START]) --> M[model 节点<br/>DeepSeek Flash]
    M -->|AIMessage.tool_calls| D{tools_condition}
    D -->|有工具调用| T[ToolNode]
    T --> W[wrap_tool_call]
    W -->|0 次| X[短路 ToolMessage / Command]
    W -->|1 次| E[execute request]
    W -->|多次| E
    E --> F[真实工具]
    F --> W
    W --> T
    T -->|ToolMessage / Command| M
    D -->|无工具调用| Z([END])
```

`ToolNode` 负责校验、注入运行时参数、执行工具以及维护工具消息协议；wrapper 包裹的是“单次工具调用的执行过程”。它可以在执行前检查或改写请求，也可以决定不执行、执行一次或执行多次，最后必须返回 `ToolMessage` 或 `Command`。


In [1]:
import importlib.metadata as metadata
import inspect
import json
import os
from dataclasses import dataclass
from pprint import pprint
from time import perf_counter
from typing import Annotated, Any, Callable, Literal, TypedDict

from dotenv import load_dotenv
from langchain.agents.middleware import wrap_tool_call as middleware_wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain.tools.tool_node import ToolCallRequest as LangChainToolCallRequest
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.prebuilt.tool_node import ToolCallRequest
from langgraph.types import Command


versions = {
    package: metadata.version(package)
    for package in (
        "langgraph",
        "langgraph-prebuilt",
        "langchain",
        "langchain-core",
        "langchain-deepseek",
    )
}
print("当前实际运行版本（完整原始字典）：")
pprint(versions)

assert versions == {
    "langgraph": "1.1.2",
    "langgraph-prebuilt": "1.0.8",
    "langchain": "1.2.12",
    "langchain-core": "1.2.18",
    "langchain-deepseek": "1.0.1",
}

load_dotenv(override=True)
deepseek_key_configured = bool(os.getenv("DEEPSEEK_API_KEY"))
print("DEEPSEEK_API_KEY：", "SET（值已隐藏）" if deepseek_key_configured else "BLOCKED（未配置）")
if not deepseek_key_configured:
    raise RuntimeError("未配置 DEEPSEEK_API_KEY，无法运行真实模型示例。")

DEEPSEEK_MODEL = "deepseek:deepseek-v4-flash"
deepseek_model = init_chat_model(
    model=DEEPSEEK_MODEL,
    temperature=0,
    extra_body={"thinking": {"type": "disabled"}},
)
print("真实模型客户端：", DEEPSEEK_MODEL)


当前实际运行版本（完整原始字典）：
{'langchain': '1.2.12',
 'langchain-core': '1.2.18',
 'langchain-deepseek': '1.0.1',
 'langgraph': '1.1.2',
 'langgraph-prebuilt': '1.0.8'}
DEEPSEEK_API_KEY： SET（值已隐藏）
真实模型客户端： deepseek:deepseek-v4-flash


## 1. 主示例：执行前改写参数，执行后记录结果

wrapper 的函数签名是：

```python
def wrapper(request: ToolCallRequest, execute: Callable) -> ToolMessage | Command:
    ...
```

本例让 DeepSeek 生成真实工具调用。wrapper 会保留原始请求，随后：

1. 把城市别名标准化为 `杭州`；
2. 从 Runtime Context 读取温度单位；
3. 补充 `include_forecast=True`；
4. 用 `request.override(...)` 创建新请求；
5. 执行工具并记录完整原始结果与耗时。


In [2]:
class WeatherState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    user_role: str


@dataclass(frozen=True)
class WeatherContext:
    preferred_units: Literal["celsius", "fahrenheit"]


weather_tool_calls: list[dict[str, Any]] = []
tool_wrapper_observations: list[dict[str, Any]] = []


@tool
def get_weather(
    city: str,
    include_forecast: bool = False,
    units: Literal["celsius", "fahrenheit"] = "celsius",
) -> dict[str, Any]:
    """查询城市天气；教学示例返回本地确定性数据。"""
    executed_args = {
        "city": city,
        "include_forecast": include_forecast,
        "units": units,
    }
    weather_tool_calls.append(executed_args)

    current_temperature = 23 if units == "celsius" else 73
    result: dict[str, Any] = {
        "city": city,
        "condition": "晴",
        "temperature": current_temperature,
        "units": units,
    }
    if include_forecast:
        result["tomorrow"] = {"condition": "多云", "temperature": current_temperature - 1}
    return result


def tool_wrapper(
    request: ToolCallRequest,
    execute: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    started_at = perf_counter()
    original_tool_call = {
        **request.tool_call,
        "args": dict(request.tool_call["args"]),
    }

    city_aliases = {
        "hangzhou": "杭州",
        "杭州": "杭州",
        "杭州市": "杭州",
    }
    rewritten_args = dict(request.tool_call["args"])
    city_key = str(rewritten_args.get("city", "")).strip().lower()
    rewritten_args["city"] = city_aliases.get(city_key, rewritten_args.get("city", ""))
    rewritten_args["include_forecast"] = True
    if request.runtime.context is not None:
        rewritten_args["units"] = request.runtime.context.preferred_units

    rewritten_call = {**request.tool_call, "args": rewritten_args}
    rewritten_request = request.override(tool_call=rewritten_call)
    result = execute(rewritten_request)

    tool_wrapper_observations.append(
        {
            "request": request,
            "original_tool_call": original_tool_call,
            "rewritten_request": rewritten_request,
            "result": result,
            "elapsed_ms": (perf_counter() - started_at) * 1000,
        }
    )
    return result


In [3]:
weather_model = deepseek_model.bind_tools([get_weather])


def call_model(state: WeatherState) -> dict[str, list[AIMessage]]:
    return {"messages": [weather_model.invoke(state["messages"])]}


weather_builder = StateGraph(WeatherState, context_schema=WeatherContext)
weather_builder.add_node("model", call_model)
weather_builder.add_node(
    "tools",
    ToolNode([get_weather], wrap_tool_call=tool_wrapper),
)
weather_builder.add_edge(START, "model")
weather_builder.add_conditional_edges(
    "model",
    tools_condition,
    {"tools": "tools", "__end__": END},
)
weather_builder.add_edge("tools", "model")
weather_graph = weather_builder.compile()

print("已编译图的完整节点映射：")
pprint(weather_graph.get_graph().nodes)
print("已编译图的完整边列表：")
pprint(weather_graph.get_graph().edges)


已编译图的完整节点映射：
{'__end__': Node(id='__end__', name='__end__', data=None, metadata=None),
 '__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None),
 'model': Node(id='model', name='model', data=model(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None),
 'tools': Node(id='tools', name='tools', data=tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'get_weather': StructuredTool(name='get_weather', description='查询城市天气；教学示例返回本地确定性数据。', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x152e9bce0>)}, _injected_args={'get_weather': _InjectedArgs(state={}, store=None, runtime=None)}, _handle_tool_errors=<function _default_handle_tool_errors at 0x150db7e20>, _messages_key='messages', _wrap_tool_call=<function t

### 首次调用

输入明确要求模型调用工具，并要求 `city` 保留英文小写，以便观察 wrapper 的参数标准化。下面先打印完整输入，再单独执行图。


In [4]:
weather_tool_calls.clear()
tool_wrapper_observations.clear()

main_input: WeatherState = {
    "messages": [
        HumanMessage(
            content=(
                "请务必调用 get_weather 工具查询杭州今天和明天的天气。"
                "调用工具时，city 参数请严格使用英文小写 hangzhou；"
                "不要自行编造结果，拿到工具结果后再用中文总结。"
            )
        )
    ],
    "user_role": "member",
}
main_context = WeatherContext(preferred_units="celsius")

print("图调用输入（完整原始 State）：")
pprint(main_input)
print("图调用 Context（完整原始对象）：")
pprint(main_context)


图调用输入（完整原始 State）：
{'messages': [HumanMessage(content='请务必调用 get_weather 工具查询杭州今天和明天的天气。调用工具时，city 参数请严格使用英文小写 hangzhou；不要自行编造结果，拿到工具结果后再用中文总结。', additional_kwargs={}, response_metadata={})],
 'user_role': 'member'}
图调用 Context（完整原始对象）：
WeatherContext(preferred_units='celsius')


In [5]:
main_result = weather_graph.invoke(main_input, context=main_context)


In [6]:
print("图调用结果（完整原始 State）：")
pprint(main_result)

print("\n按顺序输出全部原始消息：")
for index, message in enumerate(main_result["messages"]):
    print(f"\nmessages[{index}] =")
    pprint(message)

print("\nwrapper 的完整原始观察记录：")
pprint(tool_wrapper_observations)

print("\n工具实际收到的完整参数：")
pprint(weather_tool_calls)


图调用结果（完整原始 State）：
{'messages': [HumanMessage(content='请务必调用 get_weather 工具查询杭州今天和明天的天气。调用工具时，city 参数请严格使用英文小写 hangzhou；不要自行编造结果，拿到工具结果后再用中文总结。', additional_kwargs={}, response_metadata={}, id='d0548248-0856-4249-9e49-989a586e59a2'),
              AIMessage(content='我来帮你查询杭州今天和明天的天气。我会调用 get_weather 工具，并包含天气预报信息。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 368, 'total_tokens': 454, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 112}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '43b55706-51f5-4cfc-80aa-229e2cffad3c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a037db-f20f-7d11-bddb-757637d741e9-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'hangzhou', 'include_forecast': True}, 'id': '

### 结果解读

消息序列应依次包含：用户消息、DeepSeek 生成的工具调用、ToolNode 写回的 ToolMessage，以及 DeepSeek 基于工具结果生成的最终回答。`ToolCallRequest.state` 是工具执行时可见的图状态；`runtime.context` 则保存调用级配置。

wrapper 没有修改原始 `AIMessage.tool_calls`。它创建了新的 `ToolCallRequest`，因此消息历史仍保留模型最初生成的参数，而本地工具收到的是标准化后的参数。这个差异对审计尤其重要。


In [7]:
model_messages = [message for message in main_result["messages"] if isinstance(message, AIMessage)]
first_model_message = next(message for message in model_messages if message.tool_calls)
tool_messages = [message for message in main_result["messages"] if isinstance(message, ToolMessage)]
final_model_message = model_messages[-1]
main_observation = tool_wrapper_observations[0]

print("模型生成的原始 tool_calls：")
pprint(first_model_message.tool_calls)
print("wrapper 保存的原始 tool_call：")
pprint(main_observation["original_tool_call"])
print("override 后的新 tool_call：")
pprint(main_observation["rewritten_request"].tool_call)
print("工具实际执行参数：")
pprint(weather_tool_calls)
print("ToolMessage 完整列表：")
pprint(tool_messages)
print("最终 AIMessage 完整对象：")
pprint(final_model_message)
print("可选友好摘要：", final_model_message.content)


模型生成的原始 tool_calls：
[{'args': {'city': 'hangzhou', 'include_forecast': True},
  'id': 'call_00_RNu3vVGxuvaQDN3W8bxH9074',
  'name': 'get_weather',
  'type': 'tool_call'}]
wrapper 保存的原始 tool_call：
{'args': {'city': 'hangzhou', 'include_forecast': True},
 'id': 'call_00_RNu3vVGxuvaQDN3W8bxH9074',
 'name': 'get_weather',
 'type': 'tool_call'}
override 后的新 tool_call：
{'args': {'city': '杭州', 'include_forecast': True, 'units': 'celsius'},
 'id': 'call_00_RNu3vVGxuvaQDN3W8bxH9074',
 'name': 'get_weather',
 'type': 'tool_call'}
工具实际执行参数：
[{'city': '杭州', 'include_forecast': True, 'units': 'celsius'}]
ToolMessage 完整列表：
[ToolMessage(content='{"city": "杭州", "condition": "晴", "temperature": 23, "units": "celsius", "tomorrow": {"condition": "多云", "temperature": 22}}', name='get_weather', id='eef1812b-04c7-4d57-bb9c-972d467d765b', tool_call_id='call_00_RNu3vVGxuvaQDN3W8bxH9074')]
最终 AIMessage 完整对象：
AIMessage(content='根据工具返回的结果，以下是杭州的天气情况：\n\n**今天（杭州）**\n- 天气：晴\n- 气温：23°C\n\n**明天（杭州）**\n- 天气：多云\n- 气温：

In [8]:
# 机制验证与主示例、原始观察分开。
assert first_model_message.response_metadata.get("model_name") == "deepseek-v4-flash"
assert len(tool_wrapper_observations) == 1
assert len(weather_tool_calls) == 1
assert len(tool_messages) == 1
assert first_model_message.tool_calls[0] == main_observation["original_tool_call"]
assert main_observation["rewritten_request"].tool_call["args"]["city"] == "杭州"
assert weather_tool_calls[0] == {
    "city": "杭州",
    "include_forecast": True,
    "units": "celsius",
}
assert tool_messages[0].tool_call_id == first_model_message.tool_calls[0]["id"]
assert not final_model_message.tool_calls
print("主示例结构与参数改写机制验证通过。")


主示例结构与参数改写机制验证通过。


## 2. 执行控制：权限拒绝并短路

wrapper 可以完全不调用 `execute()`。这适合权限策略、缓存命中、配额耗尽或人工拒绝等场景。短路返回的 ToolMessage 仍必须使用当前 `tool_call_id`，这样模型消息与工具结果才能正确配对。


In [9]:
class ToolOnlyState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    access_level: str


protected_tool_calls: list[str] = []


@tool
def protected_lookup(record_id: str) -> str:
    """读取受保护的教学记录。"""
    protected_tool_calls.append(record_id)
    return f"record:{record_id}"


def permission_wrapper(
    request: ToolCallRequest,
    execute: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    if request.state["access_level"] != "admin":
        return ToolMessage(
            content="权限不足：该工具只允许管理员执行。",
            name=request.tool_call["name"],
            tool_call_id=request.tool_call["id"],
            status="error",
        )
    return execute(request)


permission_builder = StateGraph(ToolOnlyState)
permission_builder.add_node(
    "tools",
    ToolNode([protected_lookup], wrap_tool_call=permission_wrapper),
)
permission_builder.add_edge(START, "tools")
permission_builder.add_edge("tools", END)
permission_graph = permission_builder.compile()


In [10]:
protected_tool_calls.clear()
permission_input: ToolOnlyState = {
    "messages": [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "protected_lookup",
                    "args": {"record_id": "R-001"},
                    "id": "call-permission-denied",
                    "type": "tool_call",
                }
            ],
        )
    ],
    "access_level": "member",
}

print("权限案例输入（完整原始 State）：")
pprint(permission_input)
permission_result = permission_graph.invoke(permission_input)


权限案例输入（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'protected_lookup', 'args': {'record_id': 'R-001'}, 'id': 'call-permission-denied', 'type': 'tool_call'}], invalid_tool_calls=[])]}


In [11]:
print("权限案例输出（完整原始 State）：")
pprint(permission_result)
print("真实工具执行记录（完整原始列表）：")
pprint(protected_tool_calls)


权限案例输出（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='0bd805d7-1cdb-41e7-aadf-5e22785ebc0b', tool_calls=[{'name': 'protected_lookup', 'args': {'record_id': 'R-001'}, 'id': 'call-permission-denied', 'type': 'tool_call'}], invalid_tool_calls=[]),
              ToolMessage(content='权限不足：该工具只允许管理员执行。', name='protected_lookup', id='849b21a7-a782-45a0-9469-234f0514798c', tool_call_id='call-permission-denied', status='error')]}
真实工具执行记录（完整原始列表）：
[]


In [12]:
permission_message = permission_result["messages"][-1]
assert isinstance(permission_message, ToolMessage)
assert permission_message.status == "error"
assert permission_message.tool_call_id == "call-permission-denied"
assert protected_tool_calls == []
print("短路验证通过：ToolMessage 已写回，但真实工具没有执行。")


短路验证通过：ToolMessage 已写回，但真实工具没有执行。


## 3. 执行控制：根据错误 ToolMessage 重试

在当前 `langgraph-prebuilt==1.0.8` 中，`ToolNode` 的默认错误处理函数只自动转换 `ToolInvocationError`（例如工具参数校验错误）；普通 `TimeoutError` 会继续抛出。显式设置 `handle_tool_errors=True` 后，普通工具异常才会转换成 `status="error"` 的 ToolMessage，wrapper 可以检查返回消息状态并决定是否重试。

每次调用 `execute()` 都会再次真实调用工具。重试网络请求、付款、写数据库等有副作用的工具时，必须使用幂等键或业务去重机制。


In [13]:
retry_attempts: list[str] = []
retry_results: list[ToolMessage | Command] = []


@tool
def flaky_lookup(query: str) -> str:
    """第一次暂时失败，第二次返回本地确定性结果。"""
    retry_attempts.append(query)
    if len(retry_attempts) == 1:
        raise TimeoutError("教学用暂时性超时")
    return f"result:{query}"


def retry_wrapper(
    request: ToolCallRequest,
    execute: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    for _ in range(2):
        result = execute(request)
        retry_results.append(result)
        if not isinstance(result, ToolMessage) or result.status != "error":
            return result
    return result


retry_builder = StateGraph(ToolOnlyState)
retry_builder.add_node(
    "tools",
    ToolNode(
        [flaky_lookup],
        wrap_tool_call=retry_wrapper,
        handle_tool_errors=True,
    ),
)
retry_builder.add_edge(START, "tools")
retry_builder.add_edge("tools", END)
retry_graph = retry_builder.compile()


In [14]:
retry_attempts.clear()
retry_results.clear()
retry_input: ToolOnlyState = {
    "messages": [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "flaky_lookup",
                    "args": {"query": "langgraph"},
                    "id": "call-retry",
                    "type": "tool_call",
                }
            ],
        )
    ],
    "access_level": "member",
}

print("重试案例输入（完整原始 State）：")
pprint(retry_input)
retry_result = retry_graph.invoke(retry_input)


重试案例输入（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'flaky_lookup', 'args': {'query': 'langgraph'}, 'id': 'call-retry', 'type': 'tool_call'}], invalid_tool_calls=[])]}


In [15]:
print("重试案例输出（完整原始 State）：")
pprint(retry_result)
print("每次真实工具执行的完整记录：")
pprint(retry_attempts)
print("每次 execute 返回的完整原始结果：")
pprint(retry_results)


重试案例输出（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='1c633a0d-cef2-41b1-9eec-fa94567f3361', tool_calls=[{'name': 'flaky_lookup', 'args': {'query': 'langgraph'}, 'id': 'call-retry', 'type': 'tool_call'}], invalid_tool_calls=[]),
              ToolMessage(content='result:langgraph', name='flaky_lookup', id='5a4dc33d-e250-4c6f-b2f3-69925bfae8a9', tool_call_id='call-retry')]}
每次真实工具执行的完整记录：
['langgraph', 'langgraph']
每次 execute 返回的完整原始结果：
[ToolMessage(content="Error: TimeoutError('教学用暂时性超时')\n Please fix your mistakes.", name='flaky_lookup', tool_call_id='call-retry', status='error'),
 ToolMessage(content='result:langgraph', name='flaky_lookup', id='5a4dc33d-e250-4c6f-b2f3-69925bfae8a9', tool_call_id='call-retry')]


In [16]:
retry_message = retry_result["messages"][-1]
assert retry_attempts == ["langgraph", "langgraph"]
assert len(retry_results) == 2
assert isinstance(retry_results[0], ToolMessage) and retry_results[0].status == "error"
assert isinstance(retry_results[1], ToolMessage) and retry_results[1].status == "success"
assert isinstance(retry_message, ToolMessage) and retry_message.status == "success"
assert retry_message.tool_call_id == "call-retry"
print("重试验证通过：工具执行两次，图状态只接收 wrapper 最终返回的成功消息。")


重试验证通过：工具执行两次，图状态只接收 wrapper 最终返回的成功消息。


### 边界：关闭 ToolNode 错误转换

上一个案例显式使用 `handle_tool_errors=True`，所以 wrapper 能收到错误 ToolMessage。下面改为 `False` 并捕获完整原始异常；此时工具异常会穿过 `execute()` 与 wrapper。当前版本的默认处理函数同样会重新抛出这里的普通 `TimeoutError`。


In [17]:
always_fail_calls: list[str] = []


@tool
def always_fail(query: str) -> str:
    """始终抛出教学异常。"""
    always_fail_calls.append(query)
    raise TimeoutError("handle_tool_errors=False 原始异常")


def passthrough_wrapper(
    request: ToolCallRequest,
    execute: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    return execute(request)


raise_builder = StateGraph(ToolOnlyState)
raise_builder.add_node(
    "tools",
    ToolNode(
        [always_fail],
        wrap_tool_call=passthrough_wrapper,
        handle_tool_errors=False,
    ),
)
raise_builder.add_edge(START, "tools")
raise_builder.add_edge("tools", END)
raise_graph = raise_builder.compile()

raise_input: ToolOnlyState = {
    "messages": [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "always_fail",
                    "args": {"query": "boundary"},
                    "id": "call-raise",
                    "type": "tool_call",
                }
            ],
        )
    ],
    "access_level": "member",
}

print("关闭错误转换时的输入（完整原始 State）：")
pprint(raise_input)
try:
    raise_graph.invoke(raise_input)
except TimeoutError as error:
    print("捕获到的完整原始异常：")
    pprint(error)
else:
    raise AssertionError("预期 TimeoutError，但图调用没有抛出异常。")

print("真实工具执行记录：")
pprint(always_fail_calls)
assert always_fail_calls == ["boundary"]


关闭错误转换时的输入（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'always_fail', 'args': {'query': 'boundary'}, 'id': 'call-raise', 'type': 'tool_call'}], invalid_tool_calls=[])]}
捕获到的完整原始异常：
TimeoutError('handle_tool_errors=False 原始异常')
真实工具执行记录：
['boundary']


## 4. 执行后：规范化并过滤工具结果

wrapper 也可以处理 `execute()` 的返回值。下面的工具返回一个内部字段；wrapper 保存完整原始结果用于观察，并创建新的 ToolMessage，把内部字段替换为固定占位符后再写入图状态。生产环境中的日志仍应遵循数据分级规则，不能因为教学示例打印原始值就照搬到真实系统。


In [18]:
postprocess_observations: list[dict[str, ToolMessage]] = []


@tool
def get_profile(user_id: str) -> dict[str, str]:
    """返回本地教学用户资料。"""
    return {
        "user_id": user_id,
        "display_name": "示例用户",
        "private_note": "internal-only-demo",
    }


def redact_result_wrapper(
    request: ToolCallRequest,
    execute: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    result = execute(request)
    if not isinstance(result, ToolMessage) or not isinstance(result.content, str):
        return result

    payload = json.loads(result.content)
    payload["private_note"] = "[REDACTED]"
    processed_result = result.model_copy(
        update={"content": json.dumps(payload, ensure_ascii=False, sort_keys=True)}
    )
    postprocess_observations.append(
        {"raw_result": result, "processed_result": processed_result}
    )
    return processed_result


postprocess_builder = StateGraph(ToolOnlyState)
postprocess_builder.add_node(
    "tools",
    ToolNode([get_profile], wrap_tool_call=redact_result_wrapper),
)
postprocess_builder.add_edge(START, "tools")
postprocess_builder.add_edge("tools", END)
postprocess_graph = postprocess_builder.compile()


In [19]:
postprocess_observations.clear()
postprocess_input: ToolOnlyState = {
    "messages": [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "get_profile",
                    "args": {"user_id": "U-001"},
                    "id": "call-postprocess",
                    "type": "tool_call",
                }
            ],
        )
    ],
    "access_level": "member",
}

print("结果后处理案例输入（完整原始 State）：")
pprint(postprocess_input)
postprocess_result = postprocess_graph.invoke(postprocess_input)


结果后处理案例输入（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'get_profile', 'args': {'user_id': 'U-001'}, 'id': 'call-postprocess', 'type': 'tool_call'}], invalid_tool_calls=[])]}


In [20]:
print("结果后处理案例输出（完整原始 State）：")
pprint(postprocess_result)
print("wrapper 保存的原始结果与处理后结果：")
pprint(postprocess_observations)


结果后处理案例输出（完整原始 State）：
{'access_level': 'member',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='f1f9099b-0eea-4117-bb12-e303d4a42c8d', tool_calls=[{'name': 'get_profile', 'args': {'user_id': 'U-001'}, 'id': 'call-postprocess', 'type': 'tool_call'}], invalid_tool_calls=[]),
              ToolMessage(content='{"display_name": "示例用户", "private_note": "[REDACTED]", "user_id": "U-001"}', name='get_profile', id='f5fa7094-c442-42fb-8307-51c68ee29748', tool_call_id='call-postprocess')]}
wrapper 保存的原始结果与处理后结果：
[{'processed_result': ToolMessage(content='{"display_name": "示例用户", "private_note": "[REDACTED]", "user_id": "U-001"}', name='get_profile', id='f5fa7094-c442-42fb-8307-51c68ee29748', tool_call_id='call-postprocess'),
  'raw_result': ToolMessage(content='{"user_id": "U-001", "display_name": "示例用户", "private_note": "internal-only-demo"}', name='get_profile', tool_call_id='call-postprocess')}]


In [21]:
raw_profile_message = postprocess_observations[0]["raw_result"]
processed_profile_message = postprocess_result["messages"][-1]
raw_profile = json.loads(raw_profile_message.content)
processed_profile = json.loads(processed_profile_message.content)

assert raw_profile["private_note"] == "internal-only-demo"
assert processed_profile["private_note"] == "[REDACTED]"
assert processed_profile_message.tool_call_id == "call-postprocess"
print("结果后处理验证通过：图状态只接收过滤后的 ToolMessage。")


结果后处理验证通过：图状态只接收过滤后的 ToolMessage。


## 5. Wrapper 使用场景矩阵

| 阶段 | 使用场景 | `execute()` 次数 | 主要数据来源 | 关键风险 |
|---|---|---:|---|---|
| 执行前 | 日志、审计、计时 | 1 | `tool_call`、Runtime | 日志泄露敏感参数 |
| 执行前 | 参数格式化、别名归一 | 1 | `tool_call.args` | 必须使用 `override()`，保留原始审计证据 |
| 执行前 | 按租户、区域补充参数 | 1 | `state`、`runtime.context` | 不要把内部字段暴露到模型可见 schema |
| 执行前 | 权限、配额、限流 | 0 或 1 | State、Context、外部策略 | 短路消息仍需匹配 `tool_call_id` |
| 执行控制 | 缓存命中 | 0 | cache key、tool call | 缓存键必须覆盖所有影响结果的输入 |
| 执行控制 | 暂时性错误重试 | 多次 | ToolMessage 状态或异常 | 先明确 `handle_tool_errors`；副作用必须幂等 |
| 执行控制 | 备用参数或降级结果 | 多次或 0 次 | 错误类型、业务策略 | 不要只改工具名而保留不匹配的 `request.tool` |
| 执行控制 | 动态处理未注册工具 | 0 | `request.tool is None` | 调用 `execute()` 后仍会触发注册校验 |
| 执行后 | 结果规范化、敏感字段过滤 | 1 | ToolMessage | 保留 name、status 与 `tool_call_id` |
| 执行后 | 更新 State 或跳转 | 1 或 0 | ToolMessage、业务结果 | 返回 `Command` 时必须维护工具消息协议 |

对真实外部系统而言，wrapper 只是控制调用过程，并不会提供数据库事务、网络撤销或支付回滚。可靠性仍依赖业务侧的幂等键、去重表、超时策略与可观测性。


## 6. 两种同名 `wrap_tool_call` 的简单对比

| 对比项 | LangGraph `ToolNode.wrap_tool_call` | LangChain `@wrap_tool_call` |
|---|---|---|
| 接入方式 | 普通函数传给 `ToolNode` | 装饰函数后放入 `create_agent(middleware=[...])` |
| 使用层级 | 手写 StateGraph 的工具节点 | LangChain Agent Middleware |
| 核心请求 | `ToolCallRequest` | 同一个 `ToolCallRequest` 类型 |
| 控制能力 | 可不调用、调用一次或多次 `execute` | 可不调用、调用一次或多次 `handler` |
| 底层关系 | 直接配置 ToolNode | `create_agent` 最终把 middleware wrapper 交给内部 ToolNode |

两者的拦截思想相同，主要区别是接入层。本文只实现左侧的 LangGraph 原生路径。


In [22]:
print("LangGraph ToolNode 构造签名：")
print(inspect.signature(ToolNode))
print("\nLangChain @wrap_tool_call 装饰器签名：")
print(inspect.signature(middleware_wrap_tool_call))
print("\n两个导入路径是否指向同一个 ToolCallRequest 类型：")
print(ToolCallRequest is LangChainToolCallRequest)
print("\nToolCallRequest 的真实定义模块：")
print(ToolCallRequest.__module__)

assert ToolCallRequest is LangChainToolCallRequest


LangGraph ToolNode 构造签名：
(tools: 'Sequence[BaseTool | Callable]', *, name: 'str' = 'tools', tags: 'list[str] | None' = None, handle_tool_errors: 'bool | str | Callable[..., str] | type[Exception] | tuple[type[Exception], ...]' = <function _default_handle_tool_errors at 0x150db7e20>, messages_key: 'str' = 'messages', wrap_tool_call: 'ToolCallWrapper | None' = None, awrap_tool_call: 'AsyncToolCallWrapper | None' = None) -> 'None'

LangChain @wrap_tool_call 装饰器签名：
(func: '_CallableReturningToolResponse | None' = None, *, tools: 'list[BaseTool] | None' = None, name: 'str | None' = None) -> 'Callable[[_CallableReturningToolResponse], AgentMiddleware] | AgentMiddleware'

两个导入路径是否指向同一个 ToolCallRequest 类型：
True

ToolCallRequest 的真实定义模块：
langgraph.prebuilt.tool_node


## 结论

- `wrap_tool_call` 位于 ToolNode 的单次工具执行边界，适合统一实现观察、策略、缓存、重试与结果处理；
- 使用 `request.override(...)` 创建新请求，避免原地修改模型生成的工具调用；
- `execute()` 可以调用零次、一次或多次，但每次调用都可能产生真实副作用；
- 当前版本默认只把 `ToolInvocationError` 转成错误 ToolMessage；普通异常是否转换取决于 `handle_tool_errors`；
- 短路、改写和后处理都不能破坏 `ToolMessage.tool_call_id` 配对协议；
- 需要直接控制 StateGraph 时使用本篇的 LangGraph ToolNode 接口；使用 LangChain `create_agent` 时才选择 middleware 装饰器。
